### 실습 환경 준비

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True},
)

db = Chroma(
    persist_directory="./chroma_db_e5",
    collection_name="langchain-docs",
    embedding_function=embeddings,
    create_collection_if_not_exists=False,
)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5490.90it/s]


### HyDE(Hypothetical Document Embeddings)

In [7]:
# LangChain에서 HyDE를 사용한 RAG를 구현해 보겠다. 먼저 가상의 답변을 생성하는 Chain을 구현
model = ChatOpenAI(model="gpt-6-luna")
retriever = db.as_retriever()

prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.

문맥: """
{context}
"""

질문: {question}
''')

hypothetical_prompt = ChatPromptTemplate.from_template("""
다음 질문에 한 문장으로 답하세요.

질문: {question}
""")

hypothetical_chain = hypothetical_prompt | model | StrOutputParser()

In [8]:
# 다음으로 가상의 답변을 생성하는 Chain을 사용한 RAG의 Chain을 구현
hyde_rag_chain = {
    "question": RunnablePassthrough(),
    "context": hypothetical_chain | retriever,
} | prompt | model | StrOutputParser()

hyde_rag_chain.invoke("LangChain의 개요를 알려줘")

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


'LangChain은 대규모 언어 모델(LLM) 기반 애플리케이션을 개발하기 위한 프레임워크입니다. 모델, 임베딩, 벡터 저장소 등 다양한 구성 요소와 외부 서비스를 연결해 AI 애플리케이션을 만들 수 있습니다.\n\n주요 기능은 다음과 같습니다.\n\n- 문서 로더와 텍스트 분할기로 데이터를 처리\n- 임베딩과 벡터 저장소를 활용해 관련 정보를 검색\n- 채팅 모델과 도구를 연결해 응답 생성\n- 에이전트와 메모리를 통해 구성 요소의 실행을 조율\n\n다양한 모델과 서비스로 교체하거나 연동할 수 있으며, 생태계에는 평가·관측을 위한 LangSmith와 복잡한 에이전트 구축을 위한 LangGraph 등이 포함됩니다.'

### 복수 검색 쿼리 생성

In [11]:
# 먼저 쿼리를 생성하는 Chain(query_generation_chain)을 구현
# LangChain의 with_structured_output을 사용해 검색 쿼리의 리스트를 생성하는 예는 다음과 같다.
from pydantic import BaseModel, Field

class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="검색 쿼리 목록")

query_generation_prompt = ChatPromptTemplate.from_template("""\
질문에 대해 벡터 데이터베이스에서 관련 문서를 검색하기 위한
3개의 서로 다른 검색 쿼리를 생성하세요.
거리 기반 유사성 검색의 한계를 극복하기 위해
사용자의 질문에 대해 여러 관점을 제공하는 것이 목표입니다.

질문: {question}
""")

query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x: x.queries)
)

output = query_generation_chain.invoke({"question": "LangChain의 개요를 알려줘"})
print(output)

['LangChain이란 무엇이며 어떤 문제를 해결하는지, 주요 기능과 구성 요소를 설명하는 자료', 'LangChain의 핵심 아키텍처와 체인·프롬프트·모델·도구·메모리 등 구성 요소의 역할', 'LangChain 활용 사례와 장단점, 다른 LLM 애플리케이션 프레임워크와의 차이']


In [12]:
# query_generation_chain의 출력을 기반으로 검색하는 RAG의 Chain을 생성해 실행하는 코드는 다음과 같다.
multi_query_rag_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map()
} | prompt | model | StrOutputParser()

multi_query_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **대규모 언어 모델(LLM)을 활용한 애플리케이션과 에이전트를 만들기 위한 프레임워크**입니다. 모델, 도구, 에이전트 루프 등에 표준화된 추상화와 다양한 통합 기능을 제공해 빠르게 시작하면서도 필요에 따라 유연하게 구성할 수 있습니다.\n\n주요 특징은 다음과 같습니다.\n\n- **구성 요소 연결:** 채팅 모델, 도구, 문서 로더, 임베딩 모델, 벡터 저장소, 검색기 등을 함께 사용해 애플리케이션을 구성할 수 있습니다.\n- **외부 데이터 활용:** LLM을 다양한 데이터 소스와 연결해 관련 정보를 바탕으로 답변하게 할 수 있습니다.\n- **모델과 구성 요소 교체:** 표준 인터페이스 덕분에 구현을 크게 바꾸지 않고 다른 모델이나 구성 요소로 바꾸기 쉽습니다.\n- **에이전트 개발:** 에이전트와 자율형 애플리케이션을 빠르게 만들 때 유용하며, 복잡한 오케스트레이션이 필요하지 않은 경우에도 적합합니다.\n\nLangChain은 LangGraph 위에 구축되어 있지만, LangChain을 사용하기 위해 LangGraph를 미리 알 필요는 없습니다. 복잡한 상태 기반 워크플로나 운영 환경의 에이전트 실행이 필요하면 LangGraph와 함께 사용할 수 있습니다.'